# Mon API : modèle à adapter

Ce notebook reprend les recettes 14 à 17 de la séquence 6, dans l'ordre. Remplace tout ce qui est marqué ✏️ par les éléments de ton projet.

**Avant de commencer :**
- ta base MySQL existe, construite avec le notebook `05_base.ipynb` de la séquence 5 ;
- copie le dossier `api_modele` dans ton projet sous le nom `api`, et ce notebook dans ton dossier `notebooks` ;
- copie `README_api_modele.md` dans le dossier `api`, sous le nom `README.md`.

In [ ]:
%pip install fastapi uvicorn pymysql python-dotenv

In [ ]:
# À exécuter UNE SEULE FOIS : crée le fichier .env avec ton identifiant et ton mot de passe MySQL.
# Le mot de passe est tapé dans une case masquée : il n'apparaît jamais dans le notebook.
import os
from getpass import getpass

if os.path.exists("../.env"):
    print("Le fichier .env existe déjà : rien à faire.")
else:
    utilisateur = input("Utilisateur MySQL (souvent root) : ")
    mot_de_passe = getpass("Mot de passe MySQL : ")
    with open("../.env", "w", encoding="utf-8") as fichier:
        fichier.write(f"MYSQL_UTILISATEUR={utilisateur}\nMYSQL_MOT_DE_PASSE={mot_de_passe}\n")
    print("Fichier .env créé.")

In [ ]:
import os
import secrets
import sys

import pandas as pd
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, URL

load_dotenv("../.env")
NOM_BASE = "ma_base"                         # ✏️ le nom de ta base, le même que dans api/base.py

serveur = create_engine(URL.create("mysql+pymysql", username=os.getenv("MYSQL_UTILISATEUR"),
                                   password=os.getenv("MYSQL_MOT_DE_PASSE"), host="localhost", port=3306))
print(pd.read_sql(f"SHOW TABLES FROM {NOM_BASE}", serveur))

## 1 · Le compte de l'API et sa clé (une seule fois)

In [ ]:
if os.getenv("API_CLE"):
    print("Compte analyste et clé d'API déjà prêts.")
else:
    mot_de_passe_analyste = f"Pr-{secrets.token_hex(8)}-A9"
    with serveur.begin() as connexion:
        connexion.execute(text("DROP USER IF EXISTS 'analyste'@'localhost'"))
        connexion.execute(text("CREATE USER 'analyste'@'localhost' IDENTIFIED BY :mdp"), {"mdp": mot_de_passe_analyste})
        connexion.execute(text(f"GRANT SELECT ON {NOM_BASE}.* TO 'analyste'@'localhost'"))
    with open("../.env", "a", encoding="utf-8") as fichier:
        fichier.write(f"MYSQL_ANALYSTE_MOT_DE_PASSE={mot_de_passe_analyste}\nAPI_CLE={secrets.token_urlsafe(24)}\n")
    load_dotenv("../.env", override=True)
    print("Compte analyste créé, clé d'API rangée dans le .env.")

## 2 · Écrire les routes

✏️ Adapte `api/main.py` : une route par ressource utile à ton utilisateur, et au moins un indicateur de ta fiche projet. Puis démarre l'API.

In [ ]:
sys.path.insert(0, "../api")                # le dossier de l'API
from serveur import demarrer, arreter, ADRESSE

ENTETES = {"X-API-Key": os.getenv("API_CLE")}

def appeler(route, **parametres):
    """Envoie une demande GET à l'API et affiche son code de réponse."""
    reponse = requests.get(ADRESSE + route, params=parametres, headers=ENTETES)
    print(reponse.status_code, reponse.reason, " ", reponse.url)
    return reponse

def afficher(reponse):
    """Affiche la réponse : un tableau si c'est une liste, sinon le JSON tel quel."""
    donnees = reponse.json()
    return pd.DataFrame(donnees) if isinstance(donnees, list) else donnees

print("Outils prêts.")

demarrer()
afficher(appeler("/"))

## 3 · Tester chaque route

In [ ]:
afficher(appeler("/ma-ressource"))                    # ✏️ une cellule par route

## 4 · Avant de partager ton API

- [ ] L'API se connecte avec le compte `analyste`, en lecture seule.
- [ ] Sans clé, elle répond 401.
- [ ] Aucune route ne renvoie une donnée personnelle ligne à ligne.
- [ ] Une date ou un code impossible est refusé (422 ou 404), pas une erreur 500.
- [ ] `api/README.md` décrit chaque route.
- [ ] Le `.env` est listé dans ton `.gitignore` : vérifie qu'il n'est pas sur GitHub.

In [ ]:
arreter()